In [ ]:
'''
除了在 Euler 法的每一步之外预先计算好 L_j 相关之外
最近想到一个可以加速 H 哈密顿量部分求指数的思路，对于 Ising model 的 X-Z 分解
exp(-i t (-J XX ))=e^{itJ XX} = cos(tJ) I + i sin(tJ) XX
exp(-i t (-h Z) = e^{i t h Z} = cos(th) I + i sin(th) Z
注意在2阶 Trotter 中 t 要除以 2，并且得张量积起来
'''

In [1]:
import numpy as np
import qutip as qt
from pathlib import Path

def build_dissipative_tfim(N, J, h, gamma):
    # 1. 构建 Hamiltonian
    si_x = [
        qt.tensor([qt.qeye(2)] * i + [qt.sigmax()] + [qt.qeye(2)] * (N - i - 1))
        for i in range(N)
    ]
    si_z = [
        qt.tensor([qt.qeye(2)] * i + [qt.sigmaz()] + [qt.qeye(2)] * (N - i - 1))
        for i in range(N)
    ]

    H_x = 0
    H_z = 0
    # 相互作用项
    for i in range(N - 1):
        H_z += -J * si_z[i] * si_z[i + 1]
    # 横场项
    for i in range(N):
        H_x += -h * si_x[i]

    H = H_x + H_z

    # 2. 构建耗散算符 (Collapse Operators)
    # 这里使用 sigma_minus (自旋弛豫)
    c_ops = []
    for i in range(N):
        # sigmam() 是降算符
        L_i = qt.tensor([qt.qeye(2)] * i + [qt.sigmam()] + [qt.qeye(2)] * (N - i - 1))
        c_ops.append(np.sqrt(gamma) * L_i)  # why square root here

    return H_x, H_z, c_ops


# # 或者求解动力学演化 (Time Evolution)
# times = np.linspace(0, 10, 100)
# result = qt.mesolve(H_x + H_z, rho_ss, times, c_ops, []) # 这里用稳态做初态仅为示例

In [5]:
qt.sigmam()
qt.sigmap()

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=False
Qobj data =
[[0. 1.]
 [0. 0.]]

In [2]:
psi = qt.tensor([qt.basis(2, 0) for _ in range(5)])

# 3. 转换为密度矩阵 rho = |psi><psi|
rho_0 = qt.ket2dm(psi)
rho_0

Quantum object: dims=[[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]], shape=(32, 32), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

In [ ]:
def build_Trotter_propagator(t, r, exponential=False):
    PF = qt.to_super(qt.qeye([2] * N))
    s = t / r
    for _ in range(r):
        if exponential:
            PF = (qt.liouvillian(H_x, []) * s / 2).expm() @ PF
            PF = (qt.liouvillian(H_z, []) * s / 2).expm() @ PF
        else:
            PF = qt.to_super((-1j * H_x * s / 2).expm()) @ PF
            PF = qt.to_super((-1j * H_z * s / 2).expm()) @ PF

        for i in range(N):
            # sigmam() 是降算符
            L_i = np.sqrt(gamma) * qt.tensor(
                [qt.qeye(2)] * i + [qt.sigmam()] + [qt.qeye(2)] * (N - i - 1)
            )
            D_i = qt.lindblad_dissipator(L_i)
            if exponential:
                PF = (D_i * (s / 2)).expm() @ PF
            else:
                PF = qt.propagator(D_i, s / 2) @ PF

        for i in range(N - 1, -1, -1):
            # sigmam() 是降算符
            L_i = np.sqrt(gamma) * qt.tensor(
                [qt.qeye(2)] * i + [qt.sigmam()] + [qt.qeye(2)] * (N - i - 1)
            )
            D_i = qt.lindblad_dissipator(L_i)
            if exponential:
                PF = (D_i * (s / 2)).expm() @ PF
            else:
                PF = qt.propagator(D_i, s / 2) @ PF

        if exponential:
            PF = (qt.liouvillian(H_z, []) * s / 2).expm() @ PF
            PF = (qt.liouvillian(H_x, []) * s / 2).expm() @ PF

        else:
            PF = qt.to_super((-1j * H_z * s / 2).expm()) @ PF
            PF = qt.to_super((-1j * H_x * s / 2).expm()) @ PF
    return PF


# 使用示例
for N in range(7, 8):
    # N = 5
    gamma = 0.1

    H_x, H_z, c_ops = build_dissipative_tfim(N, J=1.0, h=0.5, gamma=0.1)

    psi = qt.tensor([qt.basis(2, 0) for _ in range(N)])

    # 转换为密度矩阵 rho = |psi><psi|
    rho_0 = qt.ket2dm(psi)

    # 求解稳态 (Steady State)
    rho_ss = qt.steadystate(H_x + H_z, c_ops)
    t = 0.2

    L = qt.liouvillian(H_x + H_z, c_ops)
    exact = (L * t).expm()

    # increase r, the supermatrix's element changes very little, but the norm really approaches the exact one
    r = 3
    PF = build_Trotter_propagator(t, r)
    rho_Trotter_no_Euler = PF @ qt.operator_to_vector(rho_0)
    rho_Trotter_no_Euler = qt.vector_to_operator(rho_Trotter_no_Euler)
    np.save(f"./data_sigmam/|000>initial/gamma_{gamma}/Trotter_Euler_10000/rho_N_{N}_r_{r}_no_Euler.npy", rho_Trotter_no_Euler.full())
    
PF

IndexError: tuple index out of range

In [ ]:
PF1 = build_Trotter_propagator(t, 1)
PF10 = build_Trotter_propagator(t, 2)
diff = PF1 - PF10
diff

Quantum object: dims=[[[2, 2, 2], [2, 2, 2]], [[2, 2, 2], [2, 2, 2]]], shape=(64, 64), type='super', dtype=Dense, isherm=False
Qobj data =
[[ 9.99609875e-04+2.27212655e-17j -9.16643985e-05-8.51196404e-04j
  -1.26795970e-03-3.39138400e-03j ...  9.11981427e-07+3.15828142e-07j
   4.53714445e-07+3.61480820e-07j -3.66136526e-08-3.34448307e-23j]
 [ 1.05642857e-06-8.62004679e-04j  3.23264580e-04-9.84466513e-04j
  -1.30565365e-05-4.74796285e-04j ...  3.82502028e-07-9.49881270e-06j
  -3.76575931e-08+9.30820930e-08j  4.67023327e-07+3.68691100e-07j]
 [-1.08627423e-03-3.42928473e-03j -1.11869116e-05-4.65429190e-04j
  -4.19929452e-04-1.82010022e-03j ... -1.92456026e-08+1.84288908e-07j
   3.73814467e-07-9.58913038e-06j  9.38832304e-07+3.20774659e-07j]
 ...
 [ 8.31562848e-06-5.63594401e-07j -4.41860427e-06-2.68166629e-05j
   1.73471055e-07+1.67583761e-06j ... -3.83306797e-04-1.89667963e-03j
  -1.50345335e-05-5.04126287e-04j -1.50899614e-03-3.48532952e-03j]
 [ 4.18187783e-06-2.88990903e-07j  3.3249234

In [ ]:
PF = build_Trotter_propagator(t, r=2)
rho_t = (PF - exact)(rho_0)
print('trace norm distance of output', rho_t.norm("tr"))
PF - exact

trace norm distance of output 0.0027550613751465528


Quantum object: dims=[[[2, 2, 2], [2, 2, 2]], [[2, 2, 2], [2, 2, 2]]], shape=(64, 64), type='super', dtype=Dense, isherm=False
Qobj data =
[[ 3.34239382e-04-2.40717771e-17j -2.93125025e-05-2.81158032e-04j
  -4.13015188e-04-1.11081930e-03j ...  3.18114823e-07+1.18829678e-07j
   1.63050394e-07+1.55500278e-07j -1.90607831e-08+1.67924608e-23j]
 [ 1.32254460e-06-2.84742390e-04j  1.06167678e-04-3.34861714e-04j
  -3.11586088e-06-1.56825856e-04j ...  2.35558949e-07-3.29109027e-06j
  -1.27441451e-08+3.24434497e-08j  1.67707850e-07+1.57109812e-07j]
 [-3.53923292e-04-1.12311438e-03j -2.50714567e-06-1.53786203e-04j
  -1.46857131e-04-6.18087002e-04j ... -4.64635064e-09+6.40750625e-08j
   2.28577985e-07-3.32061269e-06j  3.27331216e-07+1.19804535e-07j]
 ...
 [ 2.76266681e-06-7.77846021e-08j -1.11351336e-06-9.09286243e-06j
   8.16340039e-08+5.49302290e-07j ... -1.34535159e-04-6.44359596e-04j
  -3.97161842e-06-1.66419857e-04j -4.91402626e-04-1.14178807e-03j]
 [ 1.40630706e-06+5.60077344e-08j  1.6807873

In [ ]:
# 使用示例
for N in [4]:
    # N = 5
    gamma = 0.1

    H_x, H_z, c_ops = build_dissipative_tfim(N, J=1.0, h=0.5, gamma=0.1)

    psi = qt.tensor([qt.basis(2, 0) for _ in range(N)])

    # 转换为密度矩阵 rho = |psi><psi|
    rho_0 = qt.ket2dm(psi)

    # 求解稳态 (Steady State)
    rho_ss = qt.steadystate(H_x + H_z, c_ops)
    t = 0.2

    L = qt.liouvillian(H_x + H_z, c_ops)
    exact = (L * t).expm()

    # increase r, the supermatrix's element changes very little, but the norm really approaches the exact one
    r = 2
    PF = build_Trotter_propagator(t, r)

    # 将 solver 指定为 'SCS' # N=5 及以上会 die 掉
    print(qt.dnorm(exact, solver="MOSEK"))
    # print(qt.dnorm(PF, solver="MOSEK"))

    dist = qt.dnorm(PF - exact, solver='MOSEK')
    print(f"Diamond Distance: {dist}")

    # Create output directory
    dir = Path(f"./data_sigmam/gamma_{gamma}/txt")
    dir.mkdir(parents=True, exist_ok=True)

    with open(dir / f"distance_N_{N}_r_{r}.txt", "w", encoding="utf-8") as f:
        f.write(f"{dist}\n")

: 